# Sentiment downside timing placebo

**Question.** The frozen HAR sentiment overlays and the separate LSEG/Gemma firm
brake repeatedly reduce downside-squared loss. Is that reduction unusually well
timed relative to the same signal schedule placed on other dates, or is it only the
mechanical result of holding less risk?

**Scope.** This is one post-result mechanism audit, not a new strategy search. It
changes no scorer, threshold, exposure severity, holding period, breadth, cost, or
portfolio rule. It cannot promote alpha.

In [1]:
from __future__ import annotations

import hashlib
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "experiments").exists():
    ROOT = ROOT.parent
assert (ROOT / "experiments").exists(), "Run from the repository root or experiments/."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.lib.aggregators import benjamini_hochberg
from experiments.lib.fixed_share import (
    build_sparse_extreme_targets,
    evaluate_fixed_share_targets,
)
from experiments.lib.volatility_target import backtest_single_asset_exposure

SEED = 20260805
np.random.seed(SEED)

OUTPUT = ROOT / "experiments/generated/45_sentiment_downside_timing_placebo"
OUTPUT.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 100)

FNSPID_COST_BPS = 2.0
LSEG_COST_BPS = 10.0
RISK_FRACTION = 0.25
FDR_Q = 0.05


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def git_commit() -> str:
    return subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=ROOT, check=True, capture_output=True, text=True
    ).stdout.strip()


def bh_qvalues(p_values: list[float] | np.ndarray) -> np.ndarray:
    """Return monotone Benjamini-Hochberg adjusted p-values."""
    p = np.asarray(p_values, dtype=float)
    if p.ndim != 1 or len(p) == 0 or np.any(~np.isfinite(p)) or np.any((p < 0) | (p > 1)):
        raise ValueError("p-values must be a non-empty finite vector in [0, 1]")
    order = np.argsort(p, kind="mergesort")
    ranked = p[order]
    adjusted = ranked * len(p) / np.arange(1, len(p) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1].clip(0, 1)
    out = np.empty_like(adjusted)
    out[order] = adjusted
    assert np.array_equal(out <= FDR_Q, np.asarray(benjamini_hochberg(p.tolist(), q=FDR_Q)))
    return out


def path_metrics(base: pd.DataFrame, challenger: pd.DataFrame) -> dict[str, float]:
    """Return the two frozen timing estimands on aligned daily net paths."""
    left = base.loc[:, ["session_date", "net_return"]].copy()
    right = challenger.loc[:, ["session_date", "net_return"]].copy()
    for frame in (left, right):
        frame["session_date"] = pd.to_datetime(frame["session_date"]).dt.normalize()
    paired = left.merge(right, on="session_date", suffixes=("_base", "_challenger"), validate="1:1")
    if len(paired) != len(left) or len(paired) != len(right):
        raise ValueError("base and challenger paths are not fully aligned")
    base_net = paired["net_return_base"].to_numpy(dtype=float)
    challenger_net = paired["net_return_challenger"].to_numpy(dtype=float)
    base_downside = np.minimum(base_net, 0.0) ** 2
    challenger_downside = np.minimum(challenger_net, 0.0) ** 2
    return {
        "downside_reduction": float(np.mean(base_downside - challenger_downside)),
        "mean_net_difference": float(np.mean(challenger_net - base_net)),
        "base_downside_mean": float(np.mean(base_downside)),
        "challenger_downside_mean": float(np.mean(challenger_downside)),
    }


def max_path_error(actual: pd.DataFrame, expected: pd.DataFrame) -> dict[str, float]:
    """Return maximum absolute reproduction errors for shared accounting columns."""
    columns = ["net_return", "gross_return", "turnover"]
    left = actual.copy()
    right = expected.copy()
    for frame in (left, right):
        frame["session_date"] = pd.to_datetime(frame["session_date"]).dt.normalize()
    paired = left[["session_date", *columns]].merge(
        right[["session_date", *columns]], on="session_date", suffixes=("_actual", "_expected"), validate="1:1"
    )
    if len(paired) != len(left) or len(paired) != len(right):
        raise ValueError("reproduction paths do not align")
    return {
        column: float(np.max(np.abs(paired[f"{column}_actual"] - paired[f"{column}_expected"])))
        for column in columns
    }

## Frozen design

Three non-overlapping regimes enter the primary family:

1. FNSPID 2013–2019: the exact walk-forward HAR × aggregate-sentiment product;
2. FNSPID 2020–2023: the exact evaluation HAR × aggregate-sentiment product;
3. LSEG 2025–2026: the exact Gemma `strongest_event == -1` firm brake.

For each regime, shift `0` is the observed schedule. Every other possible circular
shift is enumerated exactly. A shift preserves the full risk-state sequence; for LSEG
it moves the complete 44-company flag row together, preserving firm identity, event
frequency, episode structure, and daily cross-sectional concentration. Base exposure,
returns, transaction costs, and final liquidation remain unchanged.

**Primary estimand:** mean reduction in net downside-squared return versus the local
base. The one-sided exact p-value is the fraction of all circular placements (including
observed once) at least as favourable as observed. BH-FDR is applied across the three
regimes. The cross-regime timing gate requires a positive effect and BH rejection in
all three. Mean net-return improvement is a separate secondary three-test family.

In [2]:
paths = {
    "n24_base": ROOT / "experiments/generated/24_fnspid_har_sentiment_hysteresis/evaluation_control_har_target_daily.parquet",
    "n24_overlay": ROOT / "experiments/generated/24_fnspid_har_sentiment_hysteresis/evaluation_har_sentiment_daily.parquet",
    "n25_base": ROOT / "experiments/generated/25_fnspid_har_sentiment_walkforward/walkforward_har_daily.parquet",
    "n25_overlay": ROOT / "experiments/generated/25_fnspid_har_sentiment_walkforward/walkforward_har_sentiment_daily.parquet",
    "n26_base": ROOT / "experiments/generated/26_lseg_gemma_sparse_risk_brakes/full_baseline_daily.parquet",
    "n26_overlay": ROOT / "experiments/generated/26_lseg_gemma_sparse_risk_brakes/full_firm_brake_daily.parquet",
    "gemma_firm_open": ROOT / "experiments/generated/13_lseg_44_gemma_robustness/gemma4_26b_firm_open_aggregators.parquet",
    "price_sector33": ROOT / "Data/derived/prices/lseg_us_sector_33_8m.csv",
    "price_add11": ROOT / "Data/derived/prices/lseg_us_sector_add11_8m.csv",
    "lseg_merged_manifest": ROOT / "Data/collections/lseg_us_sector_44_8m_headlines/derived/merged/manifest.json",
    "n24_manifest": ROOT / "experiments/generated/24_fnspid_har_sentiment_hysteresis/manifest.json",
    "n25_manifest": ROOT / "experiments/generated/25_fnspid_har_sentiment_walkforward/manifest.json",
    "n26_manifest": ROOT / "experiments/generated/26_lseg_gemma_sparse_risk_brakes/manifest.json",
}
for label, path in paths.items():
    assert path.exists(), f"missing {label}: {path}"

source_files = pd.DataFrame(
    [
        {"source": label, "path": str(path.relative_to(ROOT)), "sha256": sha256(path)}
        for label, path in paths.items()
    ]
)
source_files.to_csv(OUTPUT / "source_files.csv", index=False)

merged_manifest = json.loads(paths["lseg_merged_manifest"].read_text())
news_end = pd.Timestamp(merged_manifest["config"]["collection"]["end"])
price_cache_paths = sorted((ROOT / "Data/derived/prices/cache").glob("*__2025-12-26__2026-07-15.csv"))
availability = pd.DataFrame(
    [
        {
            "latest_lseg_news_boundary": news_end,
            "frozen_outcome_window_end": pd.Timestamp("2026-06-26", tz="UTC"),
            "post_window_news_available": bool(news_end > pd.Timestamp("2026-06-26", tz="UTC")),
            "price_cache_files_through_2026_07_15": len(price_cache_paths),
            "prospective_replay_run": False,
            "reason": "no local LSEG news after the frozen 2026-06-26 boundary",
        }
    ]
)
availability.to_csv(OUTPUT / "prospective_input_availability.csv", index=False)
display(availability)

,latest_lseg_news_boundary,frozen_outcome_window_end,post_window_news_available,price_cache_files_through_2026_07_15,prospective_replay_run,reason
0,2026-06-26 00:00:00+00:00,2026-06-26 00:00:00+00:00,False,33,False,no local LSEG news after the frozen 2026-06-26 boundary


The price cache extends later, but the licensed news corpus does not. Those later
price rows are therefore not a prospective sentiment sample and are not evaluated.

In [3]:
def load_daily(path: Path) -> pd.DataFrame:
    frame = pd.read_parquet(path)
    frame["session_date"] = pd.to_datetime(frame["session_date"]).dt.normalize()
    if frame["session_date"].duplicated().any():
        raise ValueError(f"duplicate daily rows in {path}")
    return frame.sort_values("session_date", kind="mergesort").reset_index(drop=True)


def fns_shift_table(
    regime: str,
    base_path: Path,
    overlay_path: Path,
) -> tuple[pd.DataFrame, dict[str, float | int]]:
    base = load_daily(base_path)
    saved_overlay = load_daily(overlay_path)
    if not base["session_date"].equals(saved_overlay["session_date"]):
        raise ValueError(f"{regime}: base and overlay dates differ")
    if (base["exposure"] <= 0).any():
        raise ValueError(f"{regime}: base exposure must be positive")
    modifier = saved_overlay["exposure"].to_numpy(dtype=float) / base["exposure"].to_numpy(dtype=float)
    if not np.all(np.isclose(modifier, 1.0) | np.isclose(modifier, RISK_FRACTION)):
        raise ValueError(f"{regime}: modifier is not the frozen 1/0.25 state")

    rows: list[dict[str, float | int | str]] = []
    observed_generated: pd.DataFrame | None = None
    returns = base.loc[:, ["session_date", "forward_return"]]
    for shift in range(len(base)):
        shifted = np.roll(modifier, shift)
        exposure = pd.DataFrame(
            {
                "session_date": base["session_date"],
                "exposure": base["exposure"].to_numpy(dtype=float) * shifted,
            }
        )
        candidate = backtest_single_asset_exposure(
            returns,
            exposure,
            cost_bps_per_side=FNSPID_COST_BPS,
        )
        if shift == 0:
            observed_generated = candidate
        metrics = path_metrics(base, candidate)
        rows.append({"regime": regime, "shift": shift, **metrics})

    assert observed_generated is not None
    reproduction = max_path_error(observed_generated, saved_overlay)
    exposure_error = float(
        np.max(np.abs(observed_generated["gross_exposure"] - saved_overlay["gross_exposure"]))
    )
    if max(*reproduction.values(), exposure_error) > 1e-12:
        raise RuntimeError(f"{regime}: observed path does not reproduce saved overlay")
    audit: dict[str, float | int] = {
        "sessions": len(base),
        "risk_off_sessions": int(np.isclose(modifier, RISK_FRACTION).sum()),
        "net_return_max_abs_error": reproduction["net_return"],
        "gross_return_max_abs_error": reproduction["gross_return"],
        "turnover_max_abs_error": reproduction["turnover"],
        "exposure_max_abs_error": exposure_error,
    }
    return pd.DataFrame(rows), audit


n25_shifts, n25_audit = fns_shift_table(
    "FNSPID 2013–2019",
    paths["n25_base"],
    paths["n25_overlay"],
)
n24_shifts, n24_audit = fns_shift_table(
    "FNSPID 2020–2023",
    paths["n24_base"],
    paths["n24_overlay"],
)

display(pd.DataFrame([{"regime": "FNSPID 2013–2019", **n25_audit}, {"regime": "FNSPID 2020–2023", **n24_audit}]))

,regime,sessions,risk_off_sessions,net_return_max_abs_error,gross_return_max_abs_error,turnover_max_abs_error,exposure_max_abs_error
0,FNSPID 2013–2019,1759,194,0.0,0.0,0.0,0.0
1,FNSPID 2020–2023,998,125,0.0,0.0,0.0,0.0


## LSEG/Gemma exact-negative firm brake

The complete 44-company flag matrix is reconstructed from the aggregate firm-open
checkpoint. No headline text is loaded. Each shift rolls the matrix by whole session
rows, so the exact same per-firm Gemma events and cross-sectional clusters are tested
at every possible calendar placement.

In [4]:
gemma = pd.read_parquet(paths["gemma_firm_open"])
gemma["session_date"] = pd.to_datetime(gemma["session_date"]).dt.normalize()
if gemma.duplicated(["session_date", "symbol"]).any():
    raise ValueError("duplicate Gemma firm-open rows")

prices = pd.concat(
    [pd.read_csv(paths["price_sector33"], parse_dates=["session_date"]), pd.read_csv(paths["price_add11"], parse_dates=["session_date"])],
    ignore_index=True,
)
prices["session_date"] = pd.to_datetime(prices["session_date"]).dt.normalize()
if prices.duplicated(["session_date", "symbol"]).any():
    raise ValueError("duplicate LSEG price rows")

open_wide = prices.pivot(index="session_date", columns="symbol", values="open").sort_index().dropna(axis="index")
sessions = pd.DatetimeIndex(sorted(gemma["session_date"].unique()))
symbols = sorted(gemma["symbol"].unique())
if len(symbols) != 44 or len(sessions) != 167:
    raise ValueError("unexpected LSEG/Gemma rectangle")
if len(sessions.difference(open_wide.index)):
    raise ValueError("Gemma sessions missing complete prices")

strongest = gemma.pivot(index="session_date", columns="symbol", values="strongest_event").reindex(
    index=sessions, columns=symbols
)
negative_flags = strongest.le(-1.0 + 1e-12)
positive_flags = pd.DataFrame(False, index=sessions, columns=symbols)


def lseg_path(flags: pd.DataFrame) -> pd.DataFrame:
    targets = build_sparse_extreme_targets(
        open_wide,
        sessions,
        flags,
        positive_flags,
        risk_fraction=RISK_FRACTION,
        entry_cost_bps=LSEG_COST_BPS,
        reallocate_to_positive=False,
    )
    return evaluate_fixed_share_targets(open_wide, targets, cost_bps_per_side=LSEG_COST_BPS)


baseline_flags = pd.DataFrame(False, index=sessions, columns=symbols)
lseg_base = lseg_path(baseline_flags)
lseg_saved_base = load_daily(paths["n26_base"])
lseg_saved_overlay = load_daily(paths["n26_overlay"])
base_reproduction = max_path_error(lseg_base, lseg_saved_base)
if max(base_reproduction.values()) > 1e-12:
    raise RuntimeError("LSEG baseline does not reproduce Notebook 26")

lseg_rows: list[dict[str, float | int | str]] = []
lseg_observed: pd.DataFrame | None = None
for shift in range(len(sessions)):
    shifted_flags = pd.DataFrame(
        np.roll(negative_flags.to_numpy(dtype=bool), shift, axis=0),
        index=sessions,
        columns=symbols,
    )
    candidate = lseg_path(shifted_flags)
    if shift == 0:
        lseg_observed = candidate
    metrics = path_metrics(lseg_base, candidate)
    lseg_rows.append({"regime": "LSEG/Gemma 2025–2026", "shift": shift, **metrics})

assert lseg_observed is not None
overlay_reproduction = max_path_error(lseg_observed, lseg_saved_overlay)
if max(overlay_reproduction.values()) > 1e-12:
    raise RuntimeError("LSEG observed brake does not reproduce Notebook 26")

lseg_audit = {
    "regime": "LSEG/Gemma 2025–2026",
    "sessions": len(sessions),
    "risk_off_sessions": int(negative_flags.any(axis=1).sum()),
    "flagged_firm_sessions": int(negative_flags.sum().sum()),
    "net_return_max_abs_error": overlay_reproduction["net_return"],
    "gross_return_max_abs_error": overlay_reproduction["gross_return"],
    "turnover_max_abs_error": overlay_reproduction["turnover"],
}
display(pd.DataFrame([lseg_audit]))

,regime,sessions,risk_off_sessions,flagged_firm_sessions,net_return_max_abs_error,gross_return_max_abs_error,turnover_max_abs_error
0,LSEG/Gemma 2025–2026,167,99,143,4.440892e-16,4.440892e-16,1.387779e-17


## Exact timing-null results

In [5]:
shift_null = pd.concat(
    [n25_shifts, n24_shifts, pd.DataFrame(lseg_rows)],
    ignore_index=True,
)
shift_null["downside_reduction_bps2"] = shift_null["downside_reduction"] * 100_000_000
shift_null["mean_net_difference_bps"] = shift_null["mean_net_difference"] * 10_000


def summarise_regime(frame: pd.DataFrame) -> dict[str, float | int | str | bool]:
    observed = frame.loc[frame["shift"].eq(0)].iloc[0]
    nonzero = frame.loc[frame["shift"].ne(0)]
    downside_p = float(np.mean(frame["downside_reduction"] >= observed["downside_reduction"] - 1e-18))
    return_p = float(np.mean(frame["mean_net_difference"] >= observed["mean_net_difference"] - 1e-18))
    return {
        "regime": str(observed["regime"]),
        "sessions": int(len(frame)),
        "nonzero_shifts": int(len(nonzero)),
        "observed_downside_reduction_bps2": float(observed["downside_reduction_bps2"]),
        "shift_mean_downside_reduction_bps2": float(nonzero["downside_reduction_bps2"].mean()),
        "observed_minus_shift_mean_downside_bps2": float(
            observed["downside_reduction_bps2"] - nonzero["downside_reduction_bps2"].mean()
        ),
        "observed_relative_downside_reduction": float(
            observed["downside_reduction"] / observed["base_downside_mean"]
        ),
        "downside_exact_p_one_sided": downside_p,
        "downside_percentile_nonzero_shifts": float(
            np.mean(nonzero["downside_reduction"] <= observed["downside_reduction"])
        ),
        "observed_mean_net_difference_bps": float(observed["mean_net_difference_bps"]),
        "shift_mean_net_difference_bps": float(nonzero["mean_net_difference_bps"].mean()),
        "observed_minus_shift_mean_net_bps": float(
            observed["mean_net_difference_bps"] - nonzero["mean_net_difference_bps"].mean()
        ),
        "return_exact_p_one_sided": return_p,
        "return_percentile_nonzero_shifts": float(
            np.mean(nonzero["mean_net_difference"] <= observed["mean_net_difference"])
        ),
    }


timing_results = pd.DataFrame(
    [summarise_regime(group) for _, group in shift_null.groupby("regime", sort=False)]
)
timing_results["downside_q_bh"] = bh_qvalues(timing_results["downside_exact_p_one_sided"].to_numpy())
timing_results["return_q_bh"] = bh_qvalues(timing_results["return_exact_p_one_sided"].to_numpy())
timing_results["downside_timing_gate"] = (
    timing_results["observed_minus_shift_mean_downside_bps2"].gt(0)
    & timing_results["downside_q_bh"].le(FDR_Q)
)
timing_results["return_timing_gate"] = (
    timing_results["observed_minus_shift_mean_net_bps"].gt(0)
    & timing_results["return_q_bh"].le(FDR_Q)
)

cross_regime_downside_gate = bool(timing_results["downside_timing_gate"].all())
cross_regime_return_gate = bool(timing_results["return_timing_gate"].all())

shift_null.to_csv(OUTPUT / "circular_shift_null.csv", index=False)
timing_results.to_csv(OUTPUT / "timing_results.csv", index=False)
reproduction_audit = pd.DataFrame(
    [
        {"regime": "FNSPID 2013–2019", **n25_audit},
        {"regime": "FNSPID 2020–2023", **n24_audit},
        lseg_audit,
    ]
)
reproduction_audit.to_csv(OUTPUT / "reproduction_audit.csv", index=False)

display(
    timing_results.style.format(
        {
            "observed_downside_reduction_bps2": "{:+.3f}",
            "shift_mean_downside_reduction_bps2": "{:+.3f}",
            "observed_minus_shift_mean_downside_bps2": "{:+.3f}",
            "observed_relative_downside_reduction": "{:+.1%}",
            "downside_exact_p_one_sided": "{:.4f}",
            "downside_q_bh": "{:.4f}",
            "observed_mean_net_difference_bps": "{:+.3f}",
            "shift_mean_net_difference_bps": "{:+.3f}",
            "return_exact_p_one_sided": "{:.4f}",
            "return_q_bh": "{:.4f}",
        }
    )
)

,regime,sessions,nonzero_shifts,observed_downside_reduction_bps2,shift_mean_downside_reduction_bps2,observed_minus_shift_mean_downside_bps2,observed_relative_downside_reduction,downside_exact_p_one_sided,downside_percentile_nonzero_shifts,observed_mean_net_difference_bps,shift_mean_net_difference_bps,observed_minus_shift_mean_net_bps,return_exact_p_one_sided,return_percentile_nonzero_shifts,downside_q_bh,return_q_bh,downside_timing_gate,return_timing_gate
0,FNSPID 2013–2019,1759,1758,+277.055,+172.356,+104.699,+16.4%,0.0455,0.955063,-0.134,-0.443,0.309451,0.1524,0.848123,0.0682,0.1524,False,False
1,FNSPID 2020–2023,998,997,+512.483,+281.284,+231.199,+21.2%,0.0100,0.990973,+0.366,-0.357,0.722971,0.0952,0.905717,0.0301,0.1428,True,False
2,LSEG/Gemma 2025–2026,167,166,+46.757,+38.374,+8.383,+2.5%,0.3653,0.638554,+0.148,-0.297,0.444725,0.0719,0.933735,0.3653,0.1428,False,False


In [6]:
regimes = timing_results["regime"].tolist()
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=False)
colors = ["#4c78a8", "#f58518", "#54a24b"]
for axis, regime, color in zip(axes, regimes, colors, strict=True):
    use = shift_null.loc[shift_null["regime"].eq(regime)]
    observed = use.loc[use["shift"].eq(0), "downside_reduction_bps2"].iloc[0]
    axis.hist(
        use.loc[use["shift"].ne(0), "downside_reduction_bps2"],
        bins=30,
        color=color,
        alpha=0.78,
    )
    axis.axvline(observed, color="black", lw=2, label="Observed timing")
    row = timing_results.loc[timing_results["regime"].eq(regime)].iloc[0]
    axis.set_title(f"{regime}\nexact p={row['downside_exact_p_one_sided']:.4f}, BH q={row['downside_q_bh']:.4f}")
    axis.set_xlabel("Downside reduction (bps²/session)")
    axis.legend(fontsize=8)
axes[0].set_ylabel("Non-zero circular shifts")
fig.suptitle("Is the observed sentiment schedule unusually good at reducing downside?", y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT / "downside_timing_shift_null.png", dpi=180, bbox_inches="tight")
plt.show()

/var/folders/dg/_nj_7w2d4mlc56b3vr3vb51m0000gn/T/ipykernel_14087/2155460256.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for axis, regime, color in zip(axes, regimes, colors, strict=True):
    use = shift_null.loc[shift_null["regime"].eq(regime)]
    null = use.loc[use["shift"].ne(0)]
    observed = use.loc[use["shift"].eq(0)].iloc[0]
    axis.scatter(
        null["downside_reduction_bps2"],
        null["mean_net_difference_bps"],
        s=14,
        alpha=0.35,
        color=color,
        label="Shifted schedules",
    )
    axis.scatter(
        observed["downside_reduction_bps2"],
        observed["mean_net_difference_bps"],
        marker="*",
        s=180,
        color="black",
        label="Observed",
        zorder=4,
    )
    axis.axhline(0, color="#777777", lw=0.8)
    axis.axvline(0, color="#777777", lw=0.8)
    axis.set_title(regime)
    axis.set_xlabel("Downside reduction (bps²/session)")
    axis.set_ylabel("Mean net-return difference (bps/session)")
    axis.legend(fontsize=8)
fig.suptitle("Downside timing versus return trade-off under every circular placement", y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT / "downside_return_timing_tradeoff.png", dpi=180, bbox_inches="tight")
plt.show()

/var/folders/dg/_nj_7w2d4mlc56b3vr3vb51m0000gn/T/ipykernel_14087/442551985.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Interpretation

In [8]:
passing_downside = int(timing_results["downside_timing_gate"].sum())
passing_return = int(timing_results["return_timing_gate"].sum())

if cross_regime_downside_gate:
    downside_conclusion = "all three regimes reject the circular timing null after BH"
elif passing_downside:
    downside_conclusion = f"{passing_downside}/3 regimes reject the circular timing null after BH"
else:
    downside_conclusion = "0/3 regimes reject the circular timing null after BH"

display(Markdown(
    "### Result\n\n"
    f"- **Primary timing family:** {downside_conclusion}.\n"
    f"- **Cross-regime downside-timing gate:** **{'PASS' if cross_regime_downside_gate else 'FAIL'}**.\n"
    f"- **Secondary return-timing family:** {passing_return}/3 local gates pass; the cross-regime gate "
    f"**{'passes' if cross_regime_return_gate else 'fails'}**.\n"
    "- **Meaning:** a lower downside-squared path is partly guaranteed when exposure is cut. Only an observed "
    "schedule that beats the same frozen state placed on other dates supports semantic timing.\n"
    "- **Claim boundary:** this post-result diagnostic may refine the risk-mechanism interpretation, but cannot "
    "promote the strategy or justify a new threshold."
))

### Result

- **Primary timing family:** 1/3 regimes reject the circular timing null after BH.
- **Cross-regime downside-timing gate:** **FAIL**.
- **Secondary return-timing family:** 0/3 local gates pass; the cross-regime gate **fails**.
- **Meaning:** a lower downside-squared path is partly guaranteed when exposure is cut. Only an observed schedule that beats the same frozen state placed on other dates supports semantic timing.
- **Claim boundary:** this post-result diagnostic may refine the risk-mechanism interpretation, but cannot promote the strategy or justify a new threshold.

In [9]:
manifest = {
    "notebook": "45_sentiment_downside_timing_placebo.ipynb",
    "status": "complete_post_result_mechanism_audit",
    "seed": SEED,
    "git_commit_at_execution": git_commit(),
    "design": {
        "primary_estimand": "mean(base net downside squared minus shifted-overlay net downside squared)",
        "secondary_estimand": "mean(shifted-overlay net return minus base net return)",
        "null": "all circular placements of each complete frozen sentiment state, observed included once",
        "primary_family": "three disjoint-regime one-sided exact downside timing tests; BH-FDR q=0.05",
        "secondary_family": "three disjoint-regime one-sided exact return timing tests; BH-FDR q=0.05",
        "cross_regime_gate": "positive observed-minus-shift-mean effect and BH rejection in all three regimes",
        "parameters_searched": False,
        "strategy_changed": False,
    },
    "regimes": [
        "FNSPID 2013-2019 walk-forward HAR times aggregate sentiment",
        "FNSPID 2020-2023 HAR times aggregate sentiment",
        "LSEG/Gemma 2025-2026 exact-negative firm brake",
    ],
    "cost_bps_per_side": {"fnspid": FNSPID_COST_BPS, "lseg": LSEG_COST_BPS},
    "risk_fraction": RISK_FRACTION,
    "input": {
        "source_sha256": {str(path.relative_to(ROOT)): sha256(path) for path in paths.values()},
        "headline_text_loaded": False,
        "licensed_text_emitted": False,
        "gemma_input_grain": "firm-open aggregate",
        "prospective_post_2026_06_26_news_available": False,
        "prospective_replay_run": False,
    },
    "reproduction": json.loads(reproduction_audit.to_json(orient="records", date_format="iso")),
    "results": json.loads(timing_results.to_json(orient="records", date_format="iso")),
    "gates": {
        "downside_local_passes": passing_downside,
        "cross_regime_downside_timing_gate": cross_regime_downside_gate,
        "return_local_passes": passing_return,
        "cross_regime_return_timing_gate": cross_regime_return_gate,
    },
    "limitations": [
        "This is a post-result mechanism audit on opened outcome windows, not confirmation.",
        "Circular shifts assume approximate stationarity and wrap sample endpoints.",
        "The FNSPID rules act on one market exposure while the LSEG/Gemma rule acts on individual firms.",
        "The LSEG exact-negative event was selected after the expanded return window was opened.",
        "A shift test distinguishes timing from generic de-risking but cannot establish causal sentiment information.",
        "No post-2026-06-26 LSEG news exists locally, so no prospective return was opened.",
    ],
}
(OUTPUT / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps(manifest["gates"], indent=2))

{
  "downside_local_passes": 1,
  "cross_regime_downside_timing_gate": false,
  "return_local_passes": 0,
  "cross_regime_return_timing_gate": false
}


## Next step

If the timing gate fails, keep the downside observation as mechanical risk reduction
rather than semantic timing. If any local test passes, report it as bounded mechanism
evidence only. In either case, do not modify the historical strategy. A real alpha test
still requires new LSEG news dates and the exact frozen prospective rule.